# Qwen3.5-0.8B × TinyCeNN PDelta3-CLVR

This Colab trains the sequential full-attention replacement experiment, verifies the exact accepted checkpoint, and uploads **only a verified model** to Hugging Face.

The long release logic lives in `scripts/qwen35_release.py`, which keeps this notebook JSON small and robust.


In [ ]:
import os, sys, pathlib, subprocess

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

REPO_DIR = pathlib.Path("/content/TinyCeNN-LM")
if not REPO_DIR.exists():
    subprocess.run(["git","clone","https://github.com/vtavakkoli/TinyCeNN-LM.git",str(REPO_DIR)], check=True)
else:
    subprocess.run(["git","-C",str(REPO_DIR),"pull","--ff-only"], check=True)

subprocess.run([
    sys.executable,"-m","pip","install","-q","-e",str(REPO_DIR),
    "transformers==4.57.6","datasets>=3,<5",
    "huggingface_hub>=0.34,<2","safetensors","pandas","matplotlib"
], check=True)

for p in (REPO_DIR/"src", REPO_DIR, REPO_DIR/"scripts"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import torch
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())
subprocess.run(["nvidia-smi"], check=False)


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = pathlib.Path("/content/drive/MyDrive/TinyCeNN-LM")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
BASE_MODEL = "Qwen/Qwen3.5-0.8B"

FEATURE_DIM = 96
LOCAL_WINDOW = 32
CHUNK_SIZE = 32
CONV_KERNEL = 4
STATE_DTYPE = "fp16"
LOCAL_GATE_INIT = 0.72

TARGET_FULL_LAYERS = 3
CONTEXT_LENGTH = 128
PROBE_CONTEXT = 128
PROBE_BLOCKS = 6
SEED = 2026

MIN_LAYER_STEPS = 60
MAX_LAYER_STEPS = 250
CHECK_EVERY = 25
LAYER_LR = 2e-4
QKV_LR_SCALE = 0.10
TEMPERATURE = 1.5

FUNCTIONAL_WEIGHT = 0.30
KL_WEIGHT = 1.00
CE_WEIGHT = 0.08
COSINE_WEIGHT = 0.20
LOCAL_GATE_PENALTY = 0.001

RESCUE_LR_SCALE = 0.50
RESCUE_FUNCTIONAL_WEIGHT = 0.15
RESCUE_KL_WEIGHT = 1.50
RESCUE_CE_WEIGHT = 0.12

ACCEPT_NMSE = 0.15
ACCEPT_COSINE = 0.94
ACCEPT_INCREMENTAL_DELTA_NLL = 0.015
ACCEPT_CUMULATIVE_DELTA_NLL = 0.05

RESUME = True
MAX_ROUNDS_PER_RUN = 2
MAX_RUNTIME_MINUTES = 240

OUTPUT_DIR = DRIVE_ROOT / f"qwen35-0.8b-pdelta3-gdn2-clvr-local{LOCAL_WINDOW}-f{FEATURE_DIM}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HF_REPO_ID = "vtava/Qwen3.5-0.8B-PDelta3-CLVR-Local32"

print("Output:", OUTPUT_DIR)
print("HF repo:", HF_REPO_ID)


## Train / resume

The pilot targets the first three full-attention layers (`3, 7, 11`). Set `TARGET_FULL_LAYERS = 6` later to continue with `15, 19, 23`.


In [ ]:
import signal

cmd = [
    sys.executable, "-u",
    str(REPO_DIR/"scripts"/"run_qwen35_pdelta3_clvr_sequential_colab.py"),
    "--base-model", BASE_MODEL,
    "--output-dir", str(OUTPUT_DIR),
    "--feature-dim", str(FEATURE_DIM),
    "--local-window", str(LOCAL_WINDOW),
    "--chunk-size", str(CHUNK_SIZE),
    "--conv-kernel", str(CONV_KERNEL),
    "--state-dtype", STATE_DTYPE,
    "--local-gate-init", str(LOCAL_GATE_INIT),
    "--target-full-layers", str(TARGET_FULL_LAYERS),
    "--context-length", str(CONTEXT_LENGTH),
    "--probe-context", str(PROBE_CONTEXT),
    "--probe-blocks", str(PROBE_BLOCKS),
    "--seed", str(SEED),
    "--min-layer-steps", str(MIN_LAYER_STEPS),
    "--max-layer-steps", str(MAX_LAYER_STEPS),
    "--check-every", str(CHECK_EVERY),
    "--layer-lr", str(LAYER_LR),
    "--qkv-lr-scale", str(QKV_LR_SCALE),
    "--temperature", str(TEMPERATURE),
    "--functional-weight", str(FUNCTIONAL_WEIGHT),
    "--kl-weight", str(KL_WEIGHT),
    "--ce-weight", str(CE_WEIGHT),
    "--cosine-weight", str(COSINE_WEIGHT),
    "--local-gate-penalty", str(LOCAL_GATE_PENALTY),
    "--rescue-lr-scale", str(RESCUE_LR_SCALE),
    "--rescue-functional-weight", str(RESCUE_FUNCTIONAL_WEIGHT),
    "--rescue-kl-weight", str(RESCUE_KL_WEIGHT),
    "--rescue-ce-weight", str(RESCUE_CE_WEIGHT),
    "--accept-nmse", str(ACCEPT_NMSE),
    "--accept-cosine", str(ACCEPT_COSINE),
    "--accept-incremental-delta-nll", str(ACCEPT_INCREMENTAL_DELTA_NLL),
    "--accept-cumulative-delta-nll", str(ACCEPT_CUMULATIVE_DELTA_NLL),
    "--max-runtime-minutes", str(MAX_RUNTIME_MINUTES),
    "--warm-start-previous-core",
    "--train-qkv",
    "--strict-acceptance",
    "--resume" if RESUME else "--no-resume",
]

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["SEQUENTIAL_MAX_ROUNDS_PER_RUN"] = str(MAX_ROUNDS_PER_RUN)
env["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

print(" ".join(cmd))
subprocess.run(cmd, env=env, check=True)


In [ ]:
import json, pandas as pd

for name in ("qwen35_run_status.json","qwen35_progress.json","qwen35_in_progress.json"):
    p = OUTPUT_DIR/name
    if p.exists():
        print("\n###", name)
        print(p.read_text()[:12000])

p = OUTPUT_DIR/"qwen35_progress.json"
if p.exists():
    reports = json.loads(p.read_text()).get("reports", [])
    if reports:
        df = pd.DataFrame(reports)
        cols = [c for c in ["layer","round","step","accepted","nmse","cosine",
                            "incremental_delta_nll","cumulative_delta_nll",
                            "local_gate_mean"] if c in df.columns]
        display(df[cols].tail(30))


## 1. Prompt + quality verification

This recomputes baseline/candidate probe NLL, checks every accepted layer's saved gates, runs deterministic prompt comparisons, and writes `qwen35_verification.json`.


In [ ]:
verify_cmd = [
    sys.executable, "-u", str(REPO_DIR/"scripts"/"qwen35_release.py"), "verify",
    "--base-model", BASE_MODEL,
    "--output-dir", str(OUTPUT_DIR),
    "--probe-context", str(PROBE_CONTEXT),
    "--probe-blocks", str(PROBE_BLOCKS),
    "--accept-nmse", str(ACCEPT_NMSE),
    "--accept-cosine", str(ACCEPT_COSINE),
    "--accept-incremental-delta-nll", str(ACCEPT_INCREMENTAL_DELTA_NLL),
    "--accept-cumulative-delta-nll", str(ACCEPT_CUMULATIVE_DELTA_NLL),
]
subprocess.run(verify_cmd, check=True)

verification = json.loads((OUTPUT_DIR/"qwen35_verification.json").read_text())
print("\nverified =", verification["verified"])
print("delta_nll =", verification["delta_nll"])


## 2. Upload verified model to Hugging Face

The upload helper refuses to publish unless verification passed. Hugging Face authentication is requested only in this cell.


In [ ]:
from huggingface_hub import get_token, login

if not get_token():
    print("Please log in with a Hugging Face WRITE token.")
    login(add_to_git_credential=False)

upload_cmd = [
    sys.executable, "-u", str(REPO_DIR/"scripts"/"qwen35_release.py"), "upload",
    "--base-model", BASE_MODEL,
    "--output-dir", str(OUTPUT_DIR),
    "--repo-id", HF_REPO_ID,
]
subprocess.run(upload_cmd, check=True)
